In [136]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
from sklearn.svm import SVR, LinearSVR
from sklearn.tree import DecisionTreeRegressor
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


if torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"
device

'cpu'

In [9]:
data = pd.read_csv('./structuralparameters-vs-capacities-5000mofs.csv', index_col=False, header=0)
data

,usablegc,usablevc,density,porosity,Ri,SSA,SPV,du,metalw,metala
0,0.2294,0.003505,1.525,0.04,2.22,632,0.026,289.0,19.449,3.704
1,0.1965,0.003082,1.566,0.02,1.77,529,0.013,289.0,19.449,3.704
2,1.3542,0.013027,0.949,0.30,5.70,3024,0.316,217.0,24.053,7.143
3,0.3773,0.004750,1.255,0.04,2.27,1002,0.032,233.0,15.330,2.439
4,0.4118,0.005179,1.253,0.04,2.41,1085,0.032,465.0,15.330,2.439
...,...,...,...,...,...,...,...,...,...,...
4995,0.1503,0.002866,1.905,0.02,2.20,377,0.011,313.0,31.873,5.882
4996,0.2563,0.002970,1.156,0.08,3.92,598,0.069,145.0,9.378,3.279
4997,0.1684,0.002585,1.533,0.06,2.48,479,0.039,201.0,24.548,2.222
4998,0.1743,0.002776,1.590,0.06,2.49,433,0.038,401.0,25.770,2.222


In [10]:
limites_tabla = pd.read_csv('./limits_5000mofs.csv', index_col=False, header=0)
limites_tabla

,Variable,Units,lowlimit,highlimit
0,usablegc,(wt.%),0.000001,15.00
1,usablevc,(kg/L),0.000001,0.06
2,density,(kg/L),0.100000,5.00
3,porosity,(dimensionless),0.000100,0.90
4,R_i,(angstroms),1.000000,40.00
5,ssa,(m^2/g),100.000000,10000.00
6,specificporevolume,(cm^3/g),0.000100,10.00
7,du,(dimensionless),0.000000,4000.00
8,metalw,%,0.000001,90.00
9,metala,%,0.000001,90.00


In [11]:
limit_inputs = np.array(limites_tabla.iloc[0:2, -2:]).T
limit_inputs

array([[1.0e-06, 1.0e-06],
       [1.5e+01, 6.0e-02]])

In [12]:
limit_outputs = np.array(limites_tabla.iloc[2:, -2:]).T
limit_outputs

array([[1.e-01, 1.e-04, 1.e+00, 1.e+02, 1.e-04, 0.e+00, 1.e-06, 1.e-06],
       [5.e+00, 9.e-01, 4.e+01, 1.e+04, 1.e+01, 4.e+03, 9.e+01, 9.e+01]])

In [59]:
escaler_X = MinMaxScaler().fit(limit_inputs)
escaler_y = MinMaxScaler().fit(limit_outputs)

In [60]:
idx_train, idx_test = train_test_split(np.arange(5000), test_size=1/3, random_state=42)

In [62]:
X = escaler_X.transform(np.array(data.iloc[:, 0:2]))
y = escaler_y.transform(np.array(data.iloc[:, 2:]))

In [89]:
type(X), type(y), X.shape, y.shape

(numpy.ndarray, numpy.ndarray, (5000, 2), (5000, 8))

In [135]:
def evaluate_with_no_torchmetrics(modelo, X, y):
    """
    Evaluación NO  usando torchmetrics (recomendado para producción)
    """
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error
    
    # Inicializar métricas
    y_predict = modelo.predict(X=X)
    
    results = {
        'mae': mean_absolute_error(y_pred=y_predict, y_true=y),
        'mse': mean_squared_error(y_pred=y_predict, y_true=y),
        'rmse': root_mean_squared_error(y_pred=y_predict, y_true=y),
        'r2': r2_score(y_pred=y_predict, y_true=y, multioutput='uniform_average'),
        'mape': mean_absolute_percentage_error(y_pred=y_predict, y_true=y)
    }
    
    return results

In [147]:
class SVM_regressor:
    def __init__(self, n_outputs=5):
        # Constructor
        self.n_outputs = n_outputs
        self.regressor = [LinearSVR(max_iter=20000) for i in range(n_outputs)]
    
    def fit(self, X, y):
        for i in range(self.n_outputs):
            self.regressor[i].fit(X, y[:, i])

    def predict(self, X):
        y = np.zeros((X.shape[0], self.n_outputs), dtype=float)
        for i in range(self.n_outputs):
            y[:, i] = self.regressor[i].predict(X)
        return y

In [148]:
regressor_svm = SVM_regressor(n_outputs=8)

In [149]:
regressor_svm.fit(X[idx_train], y[idx_train])

In [152]:
evaluate_with_no_torchmetrics(modelo=regressor_svm, X=X[idx_test], y=y[idx_test])

{'mae': 0.033231746171450825,
 'mse': 0.003218093458302214,
 'rmse': 0.047427952716798755,
 'r2': 0.472385037275271,
 'mape': 1943263194000.115}

In [153]:
class TREE_regressor:
    def __init__(self, n_outputs=5):
        # Constructor
        self.n_outputs = n_outputs
        self.regressor = [LinearSVR(max_iter=20000) for i in range(n_outputs)]
    
    def fit(self, X, y):
        for i in range(self.n_outputs):
            self.regressor[i].fit(X, y[:, i])

    def predict(self, X):
        y = np.zeros((X.shape[0], self.n_outputs), dtype=float)
        for i in range(self.n_outputs):
            y[:, i] = self.regressor[i].predict(X)
        return y

In [154]:
regressor_tree = TREE_regressor(n_outputs=8)

In [155]:
regressor_tree.fit(X[idx_train], y[idx_train])

In [156]:
evaluate_with_no_torchmetrics(modelo=regressor_tree, X=X[idx_test], y=y[idx_test])

{'mae': 0.033239139304963,
 'mse': 0.0032200068023711024,
 'rmse': 0.04742943198628102,
 'r2': 0.47239216804070827,
 'mape': 2012443012109.858}